# Day 2 | Notebook 3: Agent Memory with SemanticMessageHistory
**RedisVL Focus:** `SemanticMessageHistory`, token budgeting, TTL, multi-session management

---
## 📌 Learning Objectives
1. Understand why stateless LLMs need external memory
2. Store and retrieve conversation turns semantically (not just by recency)
3. Implement context window budget management
4. Handle multi-user sessions and TTL-based privacy compliance
---

In [2]:
# ─── INSTRUCTOR SETTINGS ─────────────────────────────────────────
SHOW_INSIGHTS = False
REDIS_URL = "redis://redis-vector-db:6379"
# ─────────────────────────────────────────────────────────────────
import time
from redisvl.extensions.message_history import SemanticMessageHistory
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("✅ Ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Ready


## 📌 Concept: The Stateless LLM Problem

Every LLM call starts fresh — it has NO memory of past conversations.
The common "fix" is to pass the full history in the prompt:

```python
prompt = f"""
Turn 1: User: 'My laptop has liquid cooling' Assistant: 'OK'
Turn 2: User: 'It has RTX 4090'              Assistant: 'OK'
Turn 3: User: 'Battery lasts 4 hours'        Assistant: 'OK'
Turn 4: User: 'Price was $2,499'             Assistant: 'OK'
Turn 5: User: 'What cooling does my laptop have?'
"""
```

**Problem**: At Turn 50, this prompt is 10,000+ tokens. Very expensive.

**Solution — Semantic Memory**:
```python
# Turn 5: retrieve ONLY relevant history
relevant = memory.get_relevant("What cooling?", top_k=2)
# Returns: Turn 1 ('liquid cooling') — only 1 turn, not 4!
```

In [11]:
# Cell 1: Create a SemanticMessageHistory for a user session
session = SemanticMessageHistory(
    name="nb03_user_alpha",       # unique session name (stored as Redis key prefix)
    redis_url=REDIS_URL,
    distance_threshold=0.35,      # similarity threshold for get_relevant()
    session_tag="user_alpha"      # metadata tag for filtering
)
print("✅ Session created: nb03_user_alpha")
print(f"   Distance threshold: {session.distance_threshold}")
print(f"   Session: {session}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Session created: nb03_user_alpha
   Distance threshold: 0.35
   Session: SemanticMessageHistory(name='nb03_user_alpha', session_tag='user_alpha', distance_threshold=0.35)


In [12]:
# Cell 2: Store a 5-turn conversation covering different topics
CONVERSATION = [
    {"role": "user",      "content": "My Aether X1 laptop has a liquid-cooled RTX 4090 GPU."},
    {"role": "assistant", "content": "Understood. Liquid cooling will keep the GPU at peak performance."},
    {"role": "user",      "content": "The OLED display is 4K 120Hz with ProMotion technology."},
    {"role": "assistant", "content": "Excellent. The ProMotion display ensures smooth visuals."},
    {"role": "user",      "content": "I paid $2,499 for it with a 3-year warranty."},
    {"role": "assistant", "content": "That is a competitive price for the specifications."},
    {"role": "user",      "content": "Battery life is around 4 hours under heavy GPU load."},
    {"role": "assistant", "content": "4 hours is expected for a high-performance GPU workload."},
    {"role": "user",      "content": "The delivery took 3 days and arrived with free accessories."},
    {"role": "assistant", "content": "Free accessories are a nice bonus for an order of this size."},
]

session.add_messages(CONVERSATION)
print(f"✅ Stored {len(CONVERSATION)} messages (5 turns)")

✅ Stored 10 messages (5 turns)


In [18]:
print(f"Total messages in the session: {session.count()}")

Total messages in the session: 10


In [19]:
corrected_context = session.get_recent()
for message in corrected_context:
    print(message)

{'role': <ChatRole.ASSISTANT: 'assistant'>, 'content': 'That is a competitive price for the specifications.'}
{'role': <ChatRole.USER: 'user'>, 'content': 'Battery life is around 4 hours under heavy GPU load.'}
{'role': <ChatRole.ASSISTANT: 'assistant'>, 'content': '4 hours is expected for a high-performance GPU workload.'}
{'role': <ChatRole.USER: 'user'>, 'content': 'The delivery took 3 days and arrived with free accessories.'}
{'role': <ChatRole.ASSISTANT: 'assistant'>, 'content': 'Free accessories are a nice bonus for an order of this size.'}


## 🔍 Demo: Semantic Retrieval — Only Relevant Turns

In [22]:
# Cell 3: Retrieve turns relevant to GPU cooling
session.set_distance_threshold(0.35)
gpu_relevant = session.get_relevant("GPU cooling system", top_k=3)

print("🔍 Query: 'GPU cooling system'")
print(f"   Retrieved {len(gpu_relevant)} relevant turns:")
for msg in gpu_relevant:
    print(f"  [{msg['role']:9}] {msg['content']}")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: Only the liquid cooling turn is returned, NOT the display or battery turns.")
    print("   Redis computes cosine similarity between the query vector and each stored turn.")
    print("   distance_threshold=0.35 filters out dissimilar turns before returning.")

🔍 Query: 'GPU cooling system'
   Retrieved 1 relevant turns:
  [ChatRole.ASSISTANT] Understood. Liquid cooling will keep the GPU at peak performance.


In [21]:
session.set_distance_threshold(0.7)
gpu_relevant = session.get_relevant("GPU cooling system", top_k=3)

print("🔍 Query: 'GPU cooling system'")
print(f"   Retrieved {len(gpu_relevant)} relevant turns:")
for msg in gpu_relevant:
    print(f"  [{msg['role']:9}] {msg['content']}")

🔍 Query: 'GPU cooling system'
   Retrieved 3 relevant turns:
  [ChatRole.ASSISTANT] Understood. Liquid cooling will keep the GPU at peak performance.
  [ChatRole.USER] My Aether X1 laptop has a liquid-cooled RTX 4090 GPU.
  [ChatRole.ASSISTANT] 4 hours is expected for a high-performance GPU workload.


In [24]:
# Cell 4: Retrieve turns relevant to price and cost
session.set_distance_threshold(0.7)
price_relevant = session.get_relevant("how much did the purchase cost", top_k=3)

print("🔍 Query: 'how much did the purchase cost'")
print(f"   Retrieved {len(price_relevant)} relevant turns:")
for msg in price_relevant:
    print(f"  [{msg['role']:9}] {msg['content']}")

🔍 Query: 'how much did the purchase cost'
   Retrieved 3 relevant turns:
  [ChatRole.USER] I paid $2,499 for it with a 3-year warranty.
  [ChatRole.ASSISTANT] That is a competitive price for the specifications.
  [ChatRole.USER] The delivery took 3 days and arrived with free accessories.


In [25]:
# Cell 5: Compare token cost — full history vs semantic retrieval
def count_tokens(messages: list) -> int:
    """Approximate token count (1 token ≈ 4 characters)."""
    return sum(len(m['content']) // 4 for m in messages)

full_history = session.get_recent(top_k=10)  # all turns
semantic_relevant = session.get_relevant("GPU cooling", top_k=3)

full_tokens = count_tokens(full_history)
semantic_tokens = count_tokens(semantic_relevant)
savings = (1 - semantic_tokens / full_tokens) * 100 if full_tokens > 0 else 0

print("📊 Token Usage Comparison:")
print(f"   Full history   : ~{full_tokens} tokens  ({len(full_history)} messages)")
print(f"   Semantic memory: ~{semantic_tokens} tokens  ({len(semantic_relevant)} messages)")
print(f"   Token savings  : {savings:.0f}%")

# Context window budget check
CONTEXT_WINDOW = 4096
SYSTEM_PROMPT_TOKENS = 200
USER_QUESTION_TOKENS = 50
available = CONTEXT_WINDOW - SYSTEM_PROMPT_TOKENS - USER_QUESTION_TOKENS - semantic_tokens
print(f"\n   Context window : {CONTEXT_WINDOW} tokens")
print(f"   Used by memory : ~{semantic_tokens} tokens")
print(f"   Available for  \n   LLM generation : ~{available} tokens ✅" if available > 500
      else f"   ⚠️  Only {available} tokens left — consider reducing top_k")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: Context window budget management prevents 'context overflow'.")
    print("   If the memory + prompt > context_window, the LLM silently truncates input.")
    print("   Always budget: window = system + memory + question + generation_reserve")

📊 Token Usage Comparison:
   Full history   : ~135 tokens  (10 messages)
   Semantic memory: ~43 tokens  (3 messages)
   Token savings  : 68%

   Context window : 4096 tokens
   Used by memory : ~43 tokens
   Available for  
   LLM generation : ~3803 tokens ✅


## 🔍 Demo: Multi-Session and Session Isolation

In [26]:
# Cell 6: Create 3 separate user sessions
sessions = {}
user_data = {
    "user_beta": "I have a Sony WH-1000XM5 headphone. The battery lasts 30 hours.",
    "user_gamma": "I use a Nebula Tab Pro tablet. The display is 12-inch OLED.",
    "user_delta": "My Aether USB-C Hub stopped working after a firmware update.",
}

for user_id, message in user_data.items():
    s = SemanticMessageHistory(
        name=f"nb03_{user_id}",
        redis_url=REDIS_URL,
        distance_threshold=0.35,
        session_tag=user_id
    )
    s.add_messages([
        {"role": "user", "content": message},
        {"role": "assistant", "content": "Thank you for sharing that information."}
    ])
    sessions[user_id] = s
    print(f"  ✅ Session created: {user_id}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✅ Session created: user_beta


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✅ Session created: user_gamma


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✅ Session created: user_delta


In [27]:
# Cell 7: Verify session isolation — user_beta cannot see user_delta's data
beta_session = sessions["user_beta"]
delta_relevant = beta_session.get_relevant("USB hub firmware", top_k=3)

print("🔍 Session Isolation Test:")
print("   Querying user_beta's session for 'USB hub firmware'")
print(f"   Results: {len(delta_relevant)} (should be 0 — user_beta has no USB hub data)")

if len(delta_relevant) == 0:
    print("   ✅ PASS: Sessions are properly isolated")
else:
    print("   ❌ FAIL: Data leak between sessions!")
    for msg in delta_relevant:
        print(f"      Leaked: {msg['content']}")

🔍 Session Isolation Test:
   Querying user_beta's session for 'USB hub firmware'
   Results: 0 (should be 0 — user_beta has no USB hub data)
   ✅ PASS: Sessions are properly isolated


In [34]:
# Cell 8: Auto-deletion
import time
from redisvl.extensions.message_history import SemanticMessageHistory

# 1. Initialize your session
short_session = SemanticMessageHistory(
    name="nb03_temp_session",
    redis_url=REDIS_URL,
    distance_threshold=0.35
)
short_session.add_messages([
    {"role": "user", "content": "This message will expire in 8 seconds."}
])

# 2. The Fix: Apply TTL using the underlying Redis client
# RedisVL exposes the active index and its client under the hood
client = short_session._index.client
prefix = short_session._index.prefix

# Find all keys matching this session's prefix and set their expiry
session_keys = client.keys(f"{prefix}:*")
for key in session_keys:
    client.expire(key, 8)  # Set 8 seconds TTL

# Check immediately
before = short_session.get_recent(top_k=5)
print(f"⏳ Immediately after store: {len(before)} message(s) found")

# Wait for expiry
print("   Waiting 9 seconds for TTL to expire...")
time.sleep(9)

# Check after TTL
after = short_session.get_recent(top_k=5)
print(f"⏳ After 9 seconds: {len(after)} message(s) found")
print("✅ TTL expired correctly" if len(after) == 0 else "❌ Data still present after TTL!")

if SHOW_INSIGHTS:
    print("\n💡 INSIGHT: TTL is essential for PDPA/GDPR compliance.")
    print("   Customer conversations must not persist indefinitely.")
    print("   Since vector message histories store each message separately,")
    print("   you must expire the keys collectively.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Immediately after store: 1 message(s) found
   Waiting 9 seconds for TTL to expire...
⏳ After 9 seconds: 0 message(s) found
✅ TTL expired correctly

💡 INSIGHT: TTL is essential for PDPA/GDPR compliance.
   Customer conversations must not persist indefinitely.
   Since vector message histories store each message separately,
   you must expire the keys collectively.


## ✅ Checkpoint

In [35]:
# Cell 9: Checkpoint
score = 0

# Test 1: semantic retrieval is selective
if len(gpu_relevant) <= 3 and len(gpu_relevant) >= 1:
    print(f"✅ Test 1 PASS: Semantic retrieval returned {len(gpu_relevant)} relevant turn(s)")
    score += 1
else:
    print(f"❌ Test 1 FAIL: Expected 1-3 turns, got {len(gpu_relevant)}")

# Test 2: token savings
if savings >= 30:
    print(f"✅ Test 2 PASS: Token savings = {savings:.0f}%")
    score += 1
else:
    print(f"❌ Test 2 FAIL: Token savings only {savings:.0f}% — add more conversation turns")

# Test 3: TTL expired
if len(after) == 0:
    print("✅ Test 3 PASS: TTL expired — session auto-deleted")
    score += 1
else:
    print("❌ Test 3 FAIL: TTL did not expire correctly")

print(f"\n🏆 Score: {score}/3")

✅ Test 1 PASS: Semantic retrieval returned 1 relevant turn(s)
✅ Test 2 PASS: Token savings = 68%
✅ Test 3 PASS: TTL expired — session auto-deleted

🏆 Score: 3/3


---
## 📝 Assignment: Multi-User Support Agent with Memory

**Scenario**: Build a customer support agent that maintains separate semantic memory for each user,
retrieves only contextually relevant past turns, and stays within a token budget.

**Task 1**: Implement `CustomerMemoryAgent` — a class that manages sessions per user.

**Task 2**: Implement `chat()` — retrieve relevant memory, build prompt, generate answer, store turn.

**Task 3 (Bonus)**: Implement `memory_stats()` — summarize what each user has discussed.

In [5]:
# ── ASSIGNMENT ────────────────────────────────────────────────────────────────
CONTEXT_WINDOW_LIMIT = 4096  # tokens
SYSTEM_PROMPT = "You are a product support agent. Answer only from provided context."

class CustomerMemoryAgent:
    """
    A customer support agent with per-user semantic memory.

    Each user has their own SemanticMessageHistory.
    Memory is retrieved semantically — not the full history.
    """

    def __init__(self, redis_url: str, session_ttl: int = 1800):
        """
        Initialize the agent.
        - Store redis_url and session_ttl
        - Keep a dict: self.sessions = {user_id: SemanticMessageHistory}
        """
        self.redis_url = redis_url
        self.session_ttl = session_ttl
        self.sessions = {}

    def _get_or_create_session(self, user_id: str):
        """
        Return existing session or create a new one.
        - New sessions should have TTL = self.session_ttl
        - Store in self.sessions[user_id]
        """
        if user_id not in self.sessions:
            ## ADD YOUR CODE HERE
            pass
            
        return self.sessions[user_id]

    def chat(self, user_id: str, user_message: str) -> dict:
        """
        Full chat pipeline.
        """
        # 1. Get/create session for user_id
        session = self._get_or_create_session(user_id)
        
        # 2. Retrieve top_k=3 relevant past turns using get_relevant()
        # (Assuming get_relevant returns a list of strings)
        memory_turns = []
        if hasattr(session, 'get_relevant'):
            ## ADD YOUR CODE HERE
            pass
        context = " | ".join(memory_turns) if memory_turns else "No prior history."
        
        # Helper function for step 3 to estimate tokens
        def count_tokens(text: str) -> int:
            return len(text.split())
            
        # 3. Count tokens: system_prompt + memory + user_message
        ## ADD YOUR CODE HERE
        
        
        # 4. If total tokens > CONTEXT_WINDOW_LIMIT: reduce top_k to 1
        if total_tokens > CONTEXT_WINDOW_LIMIT and hasattr(session, 'get_relevant'):
            ## ADD YOUR CODE HERE
            pass
            
            context = " | ".join(memory_turns) if memory_turns else "No prior history."
            total_tokens = count_tokens(SYSTEM_PROMPT) + count_tokens(context) + count_tokens(user_message)
            
        # 5 & 6. Build a simple prompt and generate answer
        answer = f"Based on your history: {context}. Answer: We are looking into your issue regarding: '{user_message}'"
        
        # 7. Store the new turn (user_message + generated answer)
        combined_turn = f"User: {user_message} | Agent: {answer}"
        if hasattr(session, 'add_message'):
            ## ADD YOUR CODE HERE
            pass
            
        # 8. Return dict
        return {
            "answer": answer,
            "tokens_used": total_tokens,
            "memory_turns_retrieved": len(memory_turns),
            "user_id": user_id
        }

    def memory_stats(self, user_id: str) -> dict:
        """
        BONUS: Return stats for a user's session
        """
        ## ADD YOUR CODE HERE
        pass


print("📝 Assignment ready — CustomerMemoryAgent implemented")

# ── EVALUATION (Run this to test your code) ─────────────────────────────────
# Assuming REDIS_URL and SemanticMessageHistory are defined above in your notebook
try:
    # REDIS_URL = "redis://localhost:6379" # Example placeholder if needed
    agent = CustomerMemoryAgent(redis_url=REDIS_URL)
    sess = agent._get_or_create_session("test_user_alpha")
    assert sess.name == "agent_test_user_alpha", "Session name must match"
    print('✅ Task 1 & 2 passed!')
except AssertionError as e:
    print(f'❌ Test failed: {e}')
except Exception as e:
    print(f'❌ Setup Error (Ensure SemanticMessageHistory is defined): {e}')

📝 Assignment ready — CustomerMemoryAgent implemented


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Task 1 & 2 passed!
